# Phase 5 — Chatbot Comparison: Voice-to-EHR Format Analysis

**Project:** EHR-Based Pre-Consultation Medical Documentation System (CSE499, NSU)
**Phase 5 Goal:** Compare how 8 voice-capable AI chatbots transcribe Bangla patient speech and structure it into an EHR format. This shows the instructor how off-the-shelf tools perform — and where our custom ASR + medical NER pipeline needs to outperform them.

## Workflow at a glance
1. A real volunteer speaks the patient script into each chatbot's voice input — once for **v1 (Pure Bangla)**, once for **v2 (Code-Mixed)**.
2. We paste the chatbot's text response back into this notebook + save a screenshot.
3. We score each chatbot on 9 criteria for both versions.
4. The notebook builds two comparison tables and a final report saved to Google Drive.

## Chatbots tested (8 — all support voice input on web or mobile)
1. **ChatGPT** (OpenAI)
2. **Gemini** (Google)
3. **Claude** (Anthropic)
4. **Perplexity**
5. **Copilot** (Microsoft)
6. **Grok** (xAI)
7. **DeepSeek**
8. **Qwen** (Alibaba / Tongyi)

> If any chatbot doesn't accept Bangla voice in its current version, fall back to typed input and clearly note this in the saved output. A failure here is itself a useful data point for the report.

**Total test runs:** 8 chatbots × 2 versions = **16 manual sessions**.


## Section 1 — Setup

Mount Google Drive and create the folder structure under `CSE499_EHR_Project/phase5_chatbot_comparison/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

ROOT = '/content/drive/MyDrive/CSE499_EHR_Project'
PHASE5_ROOT = os.path.join(ROOT, 'phase5_chatbot_comparison')

CHATBOTS = [
    'chatgpt',     # OpenAI
    'gemini',      # Google
    'claude',      # Anthropic
    'perplexity',  # Perplexity
    'copilot',     # Microsoft
    'grok',        # xAI
    'deepseek',    # DeepSeek
    'qwen',        # Alibaba (Tongyi)
]

VERSIONS = ['v1_pure_bangla', 'v2_code_mixed']

# Base folders
for f in ['prompts', 'outputs', 'screenshots', 'analysis']:
    Path(os.path.join(PHASE5_ROOT, f)).mkdir(parents=True, exist_ok=True)

# Per-chatbot output + screenshot folders
for bot in CHATBOTS:
    Path(os.path.join(PHASE5_ROOT, 'outputs', bot)).mkdir(parents=True, exist_ok=True)
    Path(os.path.join(PHASE5_ROOT, 'screenshots', bot)).mkdir(parents=True, exist_ok=True)

print('Folder structure ready at:', PHASE5_ROOT)
print()
print('Tree:')
for p in sorted(Path(PHASE5_ROOT).rglob('*')):
    if p.is_dir():
        depth = len(p.relative_to(PHASE5_ROOT).parts)
        print('  ' * depth + p.name + '/')

## Section 2 — Save Prompts to Drive

We save three text files to `prompts/`:
- `system_instruction.md` — the instruction sent to every chatbot first.
- `patient_speech_v1_pure_bangla.md` — pure Bangla (আদর্শ বাংলা).
- `patient_speech_v2_code_mixed.md` — Bangla–English code-mixed (real-patient style).

> ⚠ **Before testing:** open the two patient-speech files in Drive and replace `[NAME]` and `[AGE]` placeholders with the volunteer's actual name and age. The volunteer will then read the script aloud in front of each chatbot.

In [ ]:
SYSTEM_INSTRUCTION = """You are an AI medical documentation assistant.
A patient has just spoken the following text before their doctor's consultation.
Your job is to:
1. Extract all medically relevant information from the patient's speech.
2. Generate a structured Electronic Health Record (EHR) with the following fields:
   - Patient Name & Age
   - Chief Complaint
   - Symptoms (with duration)
   - Medical History
   - Current Medications
   - Allergies
   - Family / Contact History
3. Suggest 2-3 possible differential diagnoses based on the symptoms.
4. Flag any urgent concerns.

Do not add any information that is not present in the patient's speech.
"""

path = os.path.join(PHASE5_ROOT, 'prompts', 'system_instruction.md')
with open(path, 'w', encoding='utf-8') as f:
    f.write(SYSTEM_INSTRUCTION)
print('Saved:', path)

In [ ]:
PATIENT_V1_PURE_BANGLA = """আমার নাম [NAME], বয়স [AGE]। গত তিন সপ্তাহ ধরে কাশি হচ্ছে, রাতের বেলা বেশি হয়। সাথে জ্বর জ্বর লাগে, রাতে ঘাম হয়। খাওয়ার রুচি একদম কমে গেছে, গত এক মাসে মনে হচ্ছে একটু শুকিয়ে গেছি। বুকে মাঝে মাঝে ব্যথা করে। আমার আগে থেকে ডায়াবেটিস আছে, প্রায় পাঁচ বছর ধরে। রক্তচাপও একটু বেশি থাকে। ডায়াবেটিসের জন্য মেটফর্মিন খাই, রক্তচাপের জন্য অ্যামলোডিপিন খাই। কোনো ওষুধে অ্যালার্জি নেই আমার। আমার এক সহকর্মী গত বছর যক্ষ্মায় ভুগেছিলেন, তাঁর সাথে প্রায়ই কাজ করতাম।
"""

path = os.path.join(PHASE5_ROOT, 'prompts', 'patient_speech_v1_pure_bangla.md')
with open(path, 'w', encoding='utf-8') as f:
    f.write(PATIENT_V1_PURE_BANGLA)
print('Saved:', path)

In [ ]:
PATIENT_V2_CODE_MIXED = """আমার নাম [NAME], বয়স [AGE]। তিন সপ্তাহ ধইরা cough হইতেছে, রাইতের বেলা বেশি। সাথে একটু fever fever লাগে, রাইতে sweating হয়। Appetite একদম নাই, গত মাসে weight ও কমছে মনে হইতেছে। Chest এ একটু pain করে মাঝে মাঝে। আমার diabetes আছে, প্রায় ৫ বছর ধইরা — sugar একটু high থাকে। Blood pressure ও high, সেইটাও পুরান problem। এখন metformin আর amlodipine খাইতেছি। কোনো drug এ allergy নাই। আমার এক colleague গত বছর TB তে ভুগছিলেন, regularly contact হইছিল তার লগে।
"""

path = os.path.join(PHASE5_ROOT, 'prompts', 'patient_speech_v2_code_mixed.md')
with open(path, 'w', encoding='utf-8') as f:
    f.write(PATIENT_V2_CODE_MIXED)
print('Saved:', path)

## Section 3 — Manual Testing Procedure (read carefully)

> This is the **manual** part. The notebook can't talk to the chatbots for you. Follow this exact procedure for each of the **8 chatbots × 2 versions = 16 test runs**.

### Before you start ANY chatbot
- Open the two patient-speech files in Drive and **fill in `[NAME]` and `[AGE]`** with the volunteer's real values.
- Sign in to all 8 chatbots once at the start so you don't waste time.
- Have the volunteer beside you with the prompt file open on a phone or another screen.
- Have your screenshot tool ready (Win+Shift+S, ⌘+Shift+4, or your phone's button).

### Per-chatbot procedure (do this twice per chatbot — once for v1, once for v2)

**Step 1.** Note the **model version** shown in the chatbot UI. Examples: `GPT-5`, `Gemini 2.5 Pro`, `Claude Opus 4.7`, `DeepSeek-V3.2`, `Qwen3-Max`. Write this down — you'll need it for the saving cell.

**Step 2.** Open a **NEW chat / fresh conversation**. Never reuse a chat between tests — context bleeds between runs and ruins the comparison.

**Step 3.** **Type and send the system instruction.** Copy from `prompts/system_instruction.md`. Wait for the chatbot's acknowledgement (it will usually say "Sure, send the patient speech" or similar).

**Step 4.** **Switch to voice mode.** Have the volunteer speak the patient script:
- For **v1**: read from `patient_speech_v1_pure_bangla.md` in pure Bangla.
- For **v2**: read from `patient_speech_v2_code_mixed.md` in the code-mixed style (read English words in English pronunciation, Bangla words in Bangla).
- Speak at natural patient pace. Hesitations and small repetitions are fine — they are part of real speech and part of what we are testing.

**Step 5.** **Wait for the chatbot's full response.** Some chatbots stream slowly — don't cut it off.

**Step 6.** **Take a screenshot** of the full output. Save it to:
`CSE499_EHR_Project/phase5_chatbot_comparison/screenshots/<chatbot>/<chatbot>_v1.png` (or `_v2.png`).

**Step 7.** **Copy the entire text output** of the chatbot. You'll paste it into the corresponding notebook cell in Section 4.

### If voice input doesn't work for Bangla
Some chatbots (especially Western ones) may not transcribe Bangla speech well. If voice fails:
1. Document the failure: in the saving cell, set `voice_worked=False` and write the reason in `notes` — for example: `"Whisper transcribed Bangla as random English words"` or `"App said language not supported"`.
2. Fall back to typed input — paste the patient text from the prompt file as a typed message.
3. This is **still a valid result**. It is itself evidence that off-the-shelf voice doesn't work for Bangla — which is exactly the gap your project fills.

## Section 4 — Save Each Chatbot's Output

After running each test, paste the chatbot's response text into the relevant cell below, fill in the model version, and run the cell. The cell saves the output to Drive with metadata.

In [ ]:
from datetime import datetime

def save_chatbot_output(chatbot, version, model_version, response_text,
                        voice_worked=True, notes=''):
    """Save a chatbot response to the right folder with metadata."""
    assert chatbot in CHATBOTS, f'Unknown chatbot: {chatbot}'
    assert version in VERSIONS, f'Unknown version: {version}'

    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    voice_status = 'YES' if voice_worked else 'NO (fell back to text input)'

    header = (
        f'# {chatbot.upper()} — {version}\n\n'
        f'- **Model version:** {model_version}\n'
        f'- **Tested on:** {timestamp}\n'
        f'- **Voice input worked:** {voice_status}\n'
        f'- **Notes:** {notes if notes else "(none)"}\n\n'
        f'---\n\n'
        f'## Raw response from chatbot\n\n'
    )
    full_text = header + response_text.strip() + '\n'

    fname = f'{chatbot}_{version}.md'
    path = os.path.join(PHASE5_ROOT, 'outputs', chatbot, fname)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(full_text)
    print(f'[saved] {chatbot} / {version}  ({len(response_text)} chars)  ->  {path}')

### 4.1 — ChatGPT (OpenAI)
Paste the response into the strings below, fill in the model version, then run the cell. Both v1 and v2 are saved by one cell run.

In [ ]:
# ===== ChatGPT (OpenAI) — v1 (Pure Bangla) =====
chatgpt_v1_response = """
PASTE CHATGPT V1 (PURE BANGLA) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='chatgpt',
    version='v1_pure_bangla',
    model_version='',         # e.g. "GPT-5", "GPT-4o"
    response_text=chatgpt_v1_response,
    voice_worked=True,        # set to False if you had to fall back to typed input
    notes='',                 # any quirks, e.g. "voice mode read Bangla as English"
)

# ===== ChatGPT (OpenAI) — v2 (Code-Mixed) =====
chatgpt_v2_response = """
PASTE CHATGPT V2 (CODE-MIXED) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='chatgpt',
    version='v2_code_mixed',
    model_version='',         # e.g. "GPT-5", "GPT-4o"
    response_text=chatgpt_v2_response,
    voice_worked=True,
    notes='',
)

### 4.2 — Gemini (Google)
Paste the response into the strings below, fill in the model version, then run the cell. Both v1 and v2 are saved by one cell run.

In [ ]:
# ===== Gemini (Google) — v1 (Pure Bangla) =====
gemini_v1_response = """
PASTE GEMINI V1 (PURE BANGLA) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='gemini',
    version='v1_pure_bangla',
    model_version='',         # e.g. "Gemini 2.5 Pro", "Gemini 2.5 Flash"
    response_text=gemini_v1_response,
    voice_worked=True,        # set to False if you had to fall back to typed input
    notes='',                 # any quirks, e.g. "voice mode read Bangla as English"
)

# ===== Gemini (Google) — v2 (Code-Mixed) =====
gemini_v2_response = """
PASTE GEMINI V2 (CODE-MIXED) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='gemini',
    version='v2_code_mixed',
    model_version='',         # e.g. "Gemini 2.5 Pro", "Gemini 2.5 Flash"
    response_text=gemini_v2_response,
    voice_worked=True,
    notes='',
)

### 4.3 — Claude (Anthropic)
Paste the response into the strings below, fill in the model version, then run the cell. Both v1 and v2 are saved by one cell run.

In [ ]:
# ===== Claude (Anthropic) — v1 (Pure Bangla) =====
claude_v1_response = """
PASTE CLAUDE V1 (PURE BANGLA) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='claude',
    version='v1_pure_bangla',
    model_version='',         # e.g. "Claude Opus 4.7", "Claude Sonnet 4.6"
    response_text=claude_v1_response,
    voice_worked=True,        # set to False if you had to fall back to typed input
    notes='',                 # any quirks, e.g. "voice mode read Bangla as English"
)

# ===== Claude (Anthropic) — v2 (Code-Mixed) =====
claude_v2_response = """
PASTE CLAUDE V2 (CODE-MIXED) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='claude',
    version='v2_code_mixed',
    model_version='',         # e.g. "Claude Opus 4.7", "Claude Sonnet 4.6"
    response_text=claude_v2_response,
    voice_worked=True,
    notes='',
)

### 4.4 — Perplexity
Paste the response into the strings below, fill in the model version, then run the cell. Both v1 and v2 are saved by one cell run.

In [ ]:
# ===== Perplexity — v1 (Pure Bangla) =====
perplexity_v1_response = """
PASTE PERPLEXITY V1 (PURE BANGLA) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='perplexity',
    version='v1_pure_bangla',
    model_version='',         # e.g. "Sonar Pro", "GPT-5 (via Perplexity)"
    response_text=perplexity_v1_response,
    voice_worked=True,        # set to False if you had to fall back to typed input
    notes='',                 # any quirks, e.g. "voice mode read Bangla as English"
)

# ===== Perplexity — v2 (Code-Mixed) =====
perplexity_v2_response = """
PASTE PERPLEXITY V2 (CODE-MIXED) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='perplexity',
    version='v2_code_mixed',
    model_version='',         # e.g. "Sonar Pro", "GPT-5 (via Perplexity)"
    response_text=perplexity_v2_response,
    voice_worked=True,
    notes='',
)

### 4.5 — Copilot (Microsoft)
Paste the response into the strings below, fill in the model version, then run the cell. Both v1 and v2 are saved by one cell run.

In [ ]:
# ===== Copilot (Microsoft) — v1 (Pure Bangla) =====
copilot_v1_response = """
PASTE COPILOT V1 (PURE BANGLA) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='copilot',
    version='v1_pure_bangla',
    model_version='',         # e.g. "Copilot (GPT-5 backend)"
    response_text=copilot_v1_response,
    voice_worked=True,        # set to False if you had to fall back to typed input
    notes='',                 # any quirks, e.g. "voice mode read Bangla as English"
)

# ===== Copilot (Microsoft) — v2 (Code-Mixed) =====
copilot_v2_response = """
PASTE COPILOT V2 (CODE-MIXED) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='copilot',
    version='v2_code_mixed',
    model_version='',         # e.g. "Copilot (GPT-5 backend)"
    response_text=copilot_v2_response,
    voice_worked=True,
    notes='',
)

### 4.6 — Grok (xAI)
Paste the response into the strings below, fill in the model version, then run the cell. Both v1 and v2 are saved by one cell run.

In [ ]:
# ===== Grok (xAI) — v1 (Pure Bangla) =====
grok_v1_response = """
PASTE GROK V1 (PURE BANGLA) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='grok',
    version='v1_pure_bangla',
    model_version='',         # e.g. "Grok 4", "Grok 4 Fast"
    response_text=grok_v1_response,
    voice_worked=True,        # set to False if you had to fall back to typed input
    notes='',                 # any quirks, e.g. "voice mode read Bangla as English"
)

# ===== Grok (xAI) — v2 (Code-Mixed) =====
grok_v2_response = """
PASTE GROK V2 (CODE-MIXED) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='grok',
    version='v2_code_mixed',
    model_version='',         # e.g. "Grok 4", "Grok 4 Fast"
    response_text=grok_v2_response,
    voice_worked=True,
    notes='',
)

### 4.7 — DeepSeek
Paste the response into the strings below, fill in the model version, then run the cell. Both v1 and v2 are saved by one cell run.

In [ ]:
# ===== DeepSeek — v1 (Pure Bangla) =====
deepseek_v1_response = """
PASTE DEEPSEEK V1 (PURE BANGLA) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='deepseek',
    version='v1_pure_bangla',
    model_version='',         # e.g. "DeepSeek-V3.2", "DeepSeek-R1"
    response_text=deepseek_v1_response,
    voice_worked=True,        # set to False if you had to fall back to typed input
    notes='',                 # any quirks, e.g. "voice mode read Bangla as English"
)

# ===== DeepSeek — v2 (Code-Mixed) =====
deepseek_v2_response = """
PASTE DEEPSEEK V2 (CODE-MIXED) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='deepseek',
    version='v2_code_mixed',
    model_version='',         # e.g. "DeepSeek-V3.2", "DeepSeek-R1"
    response_text=deepseek_v2_response,
    voice_worked=True,
    notes='',
)

### 4.8 — Qwen (Alibaba)
Paste the response into the strings below, fill in the model version, then run the cell. Both v1 and v2 are saved by one cell run.

In [ ]:
# ===== Qwen (Alibaba) — v1 (Pure Bangla) =====
qwen_v1_response = """
PASTE QWEN V1 (PURE BANGLA) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='qwen',
    version='v1_pure_bangla',
    model_version='',         # e.g. "Qwen3-Max", "Qwen3-235B"
    response_text=qwen_v1_response,
    voice_worked=True,        # set to False if you had to fall back to typed input
    notes='',                 # any quirks, e.g. "voice mode read Bangla as English"
)

# ===== Qwen (Alibaba) — v2 (Code-Mixed) =====
qwen_v2_response = """
PASTE QWEN V2 (CODE-MIXED) RESPONSE HERE
"""
save_chatbot_output(
    chatbot='qwen',
    version='v2_code_mixed',
    model_version='',         # e.g. "Qwen3-Max", "Qwen3-235B"
    response_text=qwen_v2_response,
    voice_worked=True,
    notes='',
)

## Section 5 — Build Comparison Tables

Now we score each chatbot on 9 criteria, separately for v1 and v2.

**Scoring legend:**
- ✅ — Yes / Correctly handled
- ❌ — No / Failed
- ⚠️ — Partial / Some issues

> Fill in the dictionaries below by reading each chatbot's output file in `outputs/<chatbot>/`. Be honest — if it hallucinated, mark it. The instructor wants a real comparison.
>
> **Important nuance for "Hallucinated any information?"**: ✅ here means **bad** (yes, it hallucinated). ❌ means **good** (no hallucination). Keep this in mind when reading the table — or invert the wording before the demo if you prefer.

In [ ]:
EVALUATION_CRITERIA = [
    'Correctly extracted all symptoms?',
    'Correct medications identified?',
    'Allergy field correctly filled?',
    'TB contact history captured?',
    'EHR format structured properly?',
    'Differential diagnoses given?',
    'Handled Bangla / code-mixed text correctly?',
    'Hallucinated any information?',
    'Output usable by a doctor directly?',
]

In [ ]:
# === v1 (Pure Bangla) scoring ===
# Replace each '?' with one of: '✅', '❌', '⚠️'

scores_v1 = {
    'Correctly extracted all symptoms?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'Correct medications identified?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'Allergy field correctly filled?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'TB contact history captured?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'EHR format structured properly?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'Differential diagnoses given?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'Handled Bangla / code-mixed text correctly?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'Hallucinated any information?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'Output usable by a doctor directly?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
}

In [ ]:
# === v2 (Code-Mixed) scoring ===
# Replace each '?' with one of: '✅', '❌', '⚠️'

scores_v2 = {
    'Correctly extracted all symptoms?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'Correct medications identified?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'Allergy field correctly filled?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'TB contact history captured?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'EHR format structured properly?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'Differential diagnoses given?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'Handled Bangla / code-mixed text correctly?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'Hallucinated any information?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
    'Output usable by a doctor directly?': {'chatgpt':'?', 'gemini':'?', 'claude':'?', 'perplexity':'?', 'copilot':'?', 'grok':'?', 'deepseek':'?', 'qwen':'?'},
}

In [ ]:
import pandas as pd

def build_table(scores_dict, version_label):
    df = pd.DataFrame(scores_dict).T
    df = df[CHATBOTS]   # consistent column order
    df.index.name = 'Evaluation Criterion'
    print(f'\n=== Comparison Table: {version_label} ===\n')
    print(df.to_string())
    return df

df_v1 = build_table(scores_v1, 'v1 — Pure Bangla')
df_v2 = build_table(scores_v2, 'v2 — Code-Mixed')

In [ ]:
# Save comparison tables to Drive
analysis_dir = os.path.join(PHASE5_ROOT, 'analysis')

df_v1.to_csv(os.path.join(analysis_dir, 'comparison_v1_pure_bangla.csv'))
df_v2.to_csv(os.path.join(analysis_dir, 'comparison_v2_code_mixed.csv'))

with open(os.path.join(analysis_dir, 'comparison_v1_pure_bangla.md'), 'w', encoding='utf-8') as f:
    f.write('# Comparison Table — v1 Pure Bangla\n\n')
    f.write(df_v1.to_markdown())

with open(os.path.join(analysis_dir, 'comparison_v2_code_mixed.md'), 'w', encoding='utf-8') as f:
    f.write('# Comparison Table — v2 Code-Mixed\n\n')
    f.write(df_v2.to_markdown())

print('Comparison tables saved to:', analysis_dir)

## Section 6 — Final Report

This stitches everything into a single `final_report.md` you can show your instructor.

> Before running this, write your **per-chatbot observations** in the dictionary below. 1–3 sentences per chatbot, focused on: did it understand Bangla / code-mixed speech, did it follow the EHR format, any specific successes or failures.

In [ ]:
observations = {
    'chatgpt':    'Write 1-3 sentences about ChatGPT here.',
    'gemini':     'Write 1-3 sentences about Gemini here.',
    'claude':     'Write 1-3 sentences about Claude here.',
    'perplexity': 'Write 1-3 sentences about Perplexity here.',
    'copilot':    'Write 1-3 sentences about Copilot here.',
    'grok':       'Write 1-3 sentences about Grok here.',
    'deepseek':   'Write 1-3 sentences about DeepSeek here.',
    'qwen':       'Write 1-3 sentences about Qwen here.',
}

overall_conclusion = """Write a 1-2 paragraph conclusion answering:
- Which chatbot produced the most usable EHR for v1 vs v2? Why?
- How big is the performance drop from v1 to v2 (i.e. when code-mixing is introduced)?
- What does this say about the gap our project fills?
- Where does our custom ASR + medical NER pipeline need to outperform these tools?
"""

In [ ]:
report_lines = []
report_lines.append('# Phase 5 — Chatbot Comparison Report')
report_lines.append('')
report_lines.append('**Project:** EHR-Based Pre-Consultation Medical Documentation System (CSE499, NSU)')
report_lines.append(f'**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M")}')
report_lines.append('')
report_lines.append('## Setup')
report_lines.append(f'- Chatbots tested: {", ".join(c.title() for c in CHATBOTS)}')
report_lines.append('- Versions tested per chatbot: v1 (Pure Bangla), v2 (Code-Mixed)')
report_lines.append(f'- Total test runs: {len(CHATBOTS) * len(VERSIONS)}')
report_lines.append('')
report_lines.append('## System instruction sent to every chatbot')
report_lines.append('```')
report_lines.append(SYSTEM_INSTRUCTION.strip())
report_lines.append('```')
report_lines.append('')
report_lines.append('## Patient speech — v1 (Pure Bangla)')
report_lines.append('```')
report_lines.append(PATIENT_V1_PURE_BANGLA.strip())
report_lines.append('```')
report_lines.append('')
report_lines.append('## Patient speech — v2 (Code-Mixed)')
report_lines.append('```')
report_lines.append(PATIENT_V2_CODE_MIXED.strip())
report_lines.append('```')
report_lines.append('')
report_lines.append('## Comparison Table — v1 Pure Bangla')
report_lines.append('')
report_lines.append(df_v1.to_markdown())
report_lines.append('')
report_lines.append('## Comparison Table — v2 Code-Mixed')
report_lines.append('')
report_lines.append(df_v2.to_markdown())
report_lines.append('')
report_lines.append('## Per-chatbot observations')
report_lines.append('')
for bot in CHATBOTS:
    report_lines.append(f'### {bot.title()}')
    report_lines.append(observations.get(bot, '(no observations)'))
    report_lines.append('')
report_lines.append('## Overall conclusion')
report_lines.append('')
report_lines.append(overall_conclusion)
report_lines.append('')

report_path = os.path.join(PHASE5_ROOT, 'analysis', 'final_report.md')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))

print('Final report saved to:', report_path)
print()
print('=' * 60)
print('PHASE 5 — DONE')
print('=' * 60)
print()
print('Open the report:')
print(' ', report_path)
print()
print('Open the comparison CSVs:')
print(' ', os.path.join(PHASE5_ROOT, 'analysis', 'comparison_v1_pure_bangla.csv'))
print(' ', os.path.join(PHASE5_ROOT, 'analysis', 'comparison_v2_code_mixed.csv'))

## ✅ Phase 5 Complete

You should now have under `CSE499_EHR_Project/phase5_chatbot_comparison/` in Drive:

```
prompts/
  system_instruction.md
  patient_speech_v1_pure_bangla.md
  patient_speech_v2_code_mixed.md
outputs/
  chatgpt/, gemini/, claude/, perplexity/, copilot/, grok/, deepseek/, qwen/
    (each with v1.md and v2.md — 16 files total)
screenshots/
  (same 8 folders, with v1.png and v2.png — uploaded manually)
analysis/
  comparison_v1_pure_bangla.csv
  comparison_v2_code_mixed.csv
  comparison_v1_pure_bangla.md
  comparison_v2_code_mixed.md
  final_report.md
```

### What to present at the demo
1. **The two comparison tables side by side** — the v1 → v2 drop is the most important finding. It is direct evidence that off-the-shelf chatbots can't handle real Bangladeshi patient speech.
2. **2–3 example outputs side by side** (best chatbot vs worst chatbot for each version) — the screenshots make this visual.
3. **Voice-input failure rate** — how many of 8 chatbots actually accepted Bangla voice? This alone justifies your custom ASR work.
4. **Overall conclusion** — link the findings back to your project's contribution: custom Bangla ASR + medical NER + EHR formatting, all in one pipeline tuned for actual clinic speech.
